In [20]:
import sys
sys.path.append("../")
from src.embeddings import TransformerEmbeddings
from src.model import MultiHeadAttention
import torch 
import torch.nn as nn


In [25]:
# create a dummy input for tensor shape pass check 
def get_random_embedding(vocab_size=100, d_model=64):

    x = torch.randint(0, vocab_size, (8, 10)) # (batch_size, seq_len)
    embeddings = TransformerEmbeddings(vocab_size, d_model)
    x = embeddings(x) # (batch_size, seq_len, d_model)
    return x

In [ ]:
Q = get_random_embedding()
K = get_random_embedding()
V = get_random_embedding()

print(f"shape of Q: {Q.shape}\n") # (8, 10, 64)

mha = MultiHeadAttention(d_model=64, n_heads=4)
output = mha(Q, K, V)

print(f"shape of output after mha: {output.shape}") # (8, 10, 64)

shape of Q: torch.Size([8, 10, 64])

shape of output after mha: torch.Size([8, 10, 64])


In [27]:
# test for masked attention

batch_size, seq_len, d_model, n_heads = 8, 10, 64, 4

Q = get_random_embedding(vocab_size=100, d_model=d_model)
K = get_random_embedding(vocab_size=100, d_model=d_model)
V = get_random_embedding(vocab_size=100, d_model=d_model)

# Causal mask: lower-triangular matrix of 1s
# Shape: (1, 1, seq_len, seq_len) — broadcasts over batch & heads
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)
print(f"Causal mask shape: {causal_mask.shape}")
print(f"Causal mask:\n{causal_mask[0, 0]}\n")

mha = MultiHeadAttention(d_model=d_model, n_heads=n_heads)

# Without mask — all positions can attend to each other
output_no_mask = mha(Q, K, V)
print(f"Output (no mask) shape: {output_no_mask.shape}")

# With causal mask — position i can only attend to positions 0..i
output_masked = mha(Q, K, V, mask=causal_mask)
print(f"Output (causal mask) shape: {output_masked.shape}")

# They should differ since masking changes the attention pattern
print(f"\nOutputs are different: {not torch.allclose(output_no_mask, output_masked)}")


Causal mask shape: torch.Size([1, 1, 10, 10])
Causal mask:
tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])

Output (no mask) shape: torch.Size([8, 10, 64])
Output (causal mask) shape: torch.Size([8, 10, 64])

Outputs are different: True
